# Day 4 — Spatial Graph Construction
Goal: turn the dataset into a graph where each house is a node connected to its
K nearest neighbors by physical (Haversine) distance. This graph is what Day 5's
attention model will operate on.


In [5]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

train = pd.read_csv('train_split.csv')
test = pd.read_csv('test_split.csv')
train['split'] = 'train'
test['split'] = 'test'
full = pd.concat([train, test], ignore_index=True)

K = 10
coords_rad = np.radians(full[['lat','long']].values)

# haversine metric in sklearn expects radians and returns distance in radians;
# multiply by Earth's radius (km) to get real distances
nn = NearestNeighbors(n_neighbors=K+1, metric='haversine').fit(coords_rad)
dist, idx = nn.kneighbors(coords_rad)

EARTH_RADIUS_KM = 6371.0
dist_km = dist * EARTH_RADIUS_KM

# drop the first neighbor (each point is its own nearest neighbor)
neighbor_idx = idx[:, 1:]
neighbor_dist = dist_km[:, 1:]

print(neighbor_idx.shape, neighbor_dist.shape)
print("Average neighbour distance (km):", np.mean(neighbor_dist))
print("Maximum neighbour distance (km):", np.max(neighbor_dist))
print("Median distance to 10th nearest neighbor (km):", np.median(neighbor_dist[:, -1]))


(21435, 10) (21435, 10)
Average neighbour distance (km): 0.2808199806111431
Maximum neighbour distance (km): 24.238627760286462
Median distance to 10th nearest neighbor (km): 0.33212099806513773


In [6]:
np.save('neighbor_idx.npy', neighbor_idx)
np.save('neighbor_dist.npy', neighbor_dist)
full.to_csv('full_with_index.csv', index=False)  # positional index == row order used above


In [7]:
import folium

m = folium.Map(location=[47.55, -122.2], zoom_start=10, tiles='cartodbpositron')
sample_ids = np.random.RandomState(1).choice(len(full), 6, replace=False)

colors = ['red','blue','green','purple','orange','darkred']
for c, i in zip(colors, sample_ids):
    row = full.iloc[i]
    folium.Marker([row['lat'], row['long']], icon=folium.Icon(color=c),
                  popup=f"House {i} (${row['price']:,.0f})").add_to(m)
    for j in neighbor_idx[i]:
        nb = full.iloc[j]
        folium.CircleMarker([nb['lat'], nb['long']], radius=4, color=c, fill=True).add_to(m)
        folium.PolyLine([[row['lat'], row['long']], [nb['lat'], nb['long']]],
                         color=c, weight=1, opacity=0.5).add_to(m)

m.save('knn_graph_sample.html')
m


## Day 4 Summary

- Constructed a spatial K-Nearest Neighbour (KNN) graph (K=10) using Haversine distance.
- Represented each house as a graph node connected to its geographically closest neighbours.
- Verified neighbour connectivity through interactive Folium visualisation.
- Saved neighbour indices and distances for downstream Graph Neural Network modelling.
- The resulting graph captures spatial relationships that traditional tabular models cannot explicitly learn.
